[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HernanBustamanteG/lapis26-p3-corrientes/blob/main/notebooks/00_entorno.ipynb)

# 00 · Entorno y comprobaciones

**XII LAPIS 2026 · Proyecto 3 (J. N. Garavito-Camargo): corrientes estelares como sondas de la LMC y del
movimiento reflejo de la Vía Láctea.** Hernán Bustamante Gallagher.

Este notebook prepara el entorno y corre las pruebas del paquete `p3streams` antes de confiar en cualquier
resultado. Orden de los notebooks:

| notebook | qué hace | tiempo en Colab |
|---|---|---|
| 00 | entorno, versiones, autotest | ~2 min |
| 01 | hito 1: una corriente en la MW aislada y en MW + LMC | ~2 min |
| 02 | hito 2: los cuatro potenciales validados, $\lvert\mathbf a_{\rm MW}(t)\rvert$ y distancia de la LMC | ~4 min |
| 03 | hito 3: la corriente y sus tres métricas en los cuatro modelos; corrientes corta, larga y de dos vueltas | ~3 min |
| 04 | hito 4: el catálogo (lee el incluido; puede rehacerlo) | 1 min (o ~1.5 h) |
| 05 | hito 5: clasificación reflejo / torque y tendencias | ~1 min |
| 06 | hito 5: predicciones para LSST | ~2 min |
| 07 | hito 6: corrientes observadas; masa de la LMC | ~2 min |
| 08 | figuras finales (académicas y de prensa) | ~3 min |
| 09 | videos (MP4 y GIF) | ~10 min |

**Cómo funciona en Colab.** La primera celda de cada notebook clona este repositorio, instala versiones fijas
(`requirements-colab.txt`: gala 1.12.0 y numpy 2.2.x, porque numpy 2.3 o mayor rompe numba, que viene en Colab)
y monta Google Drive para guardar las salidas en `MyDrive/lapis26_p3/salidas/`. Si numpy cambia de versión la
sesión se reinicia una sola vez: hay que volver a ejecutar la primera celda.

**Reglas de las salidas.** Cada notebook escribe en una carpeta nueva `salidas/<notebook>_<fecha>/`; nada se
pisa. Los datos de entrada (`datos_entrada/`) son de solo lectura.

In [1]:
# Celda 1: entorno. En Colab clona el repositorio, instala versiones fijas y monta Google Drive (las salidas
# quedan en MyDrive/lapis26_p3/salidas). Si numpy cambia de version, la sesion se reinicia UNA vez:
# vuelve a ejecutar esta celda. En una maquina propia solo comprueba versiones.
import os, sys, subprocess
REPO_URL = "https://github.com/HernanBustamanteG/lapis26-p3-corrientes.git"
USAR_DRIVE = True
if "google.colab" in sys.modules:
    RAIZ = "/content/lapis26-p3-corrientes"
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "-q", REPO_URL, RAIZ], check=True)
else:
    RAIZ = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, RAIZ)
os.chdir(RAIZ)
from p3streams import entorno
if entorno.en_colab():
    entorno.preparar_colab(RAIZ, montar_drive=USAR_DRIVE)
entorno.resumen();

Mounted at /content/drive
Python 3.13.18
  numpy       2.2.6
  scipy       1.16.3
  astropy     7.2.3
  gala        1.12.0
  matplotlib  3.10.0
  pandas      2.2.3
  healpy      1.20.1
  galstreams  1.2.1
Salidas en: /content/drive/MyDrive/lapis26_p3/salidas
ffmpeg: /usr/bin/ffmpeg | CPU: 2
Entorno OK


## Autotest

Casos con respuesta conocida: la curva de rotación de la MW del tutorial, polos y planos de un anillo,
el desalineamiento de una corriente sintética (0° y 20°), el ángulo desenrollado, $f_{\rm reflex}$, la isócrona,
los errores de LSST, el campo uniforme contra $-\mathbf a_{\rm MW}$, el modelo D contra una integración en el
marco inercial, la órbita del enunciado (pericentro ~27 kpc, e ~0.6) y la reproducibilidad con la semilla.

In [2]:
from p3streams import autotest
OK = autotest.correr()
assert OK, 'alguna prueba falla: no seguir'

AUTOTEST p3streams (p3streams 1.0): no escribe nada.
  [PASA] MW del tutorial: v_c(8.1 kpc) = 229.4 km/s
  [PASA] polo medio de un anillo en z = 0 es +z
  [PASA] normal SVD y espesor (0.1 kpc)
  [PASA] desalineamiento local de una corriente sintetica = 0 deg
  [PASA] desalineamiento local de una corriente sintetica = 20 deg
  [PASA] angulo desenrollado hasta una vuelta por brazo
  [PASA] f_reflex = 3/(3+1) = 0.75 -> reflex
  [PASA] isocrona PARSEC: 1.61 estrellas por Msun, M_r min -3.3
  [PASA] sigma_mu LSST: 0.2 (r=21), 1.0 (r=24) mas/yr a 10 anios y escala t^-1.5
  [PASA] campo uniforme = -a_MW (error relativo 4.0e-04 a <= 200 kpc)
  [PASA] modelo D = marco inercial para una particula de prueba (|dx| max 5.4e-04 kpc en 1 Gyr)
  [PASA] f = 0.5 a 100 kpc: peri 26.7 kpc, e = 0.58 (enunciado: ~27 kpc, ~0.6)
  [PASA] misma semilla -> misma corriente
  [PASA] modelo A: polo e inclinacion ~0 (0.05 y 0.79 deg)
Tiempo: 41 s
Autotest: 14 de 14 pruebas pasan.


## Decisiones del proyecto

Todas están en `p3streams/config.py`, con su fuente. Las principales:

In [3]:
from p3streams import config as C
print("MW (tutorial):  disco MN3 %.3g Msun, bulbo %.3g, nucleo %.3g, halo NFW v_c = %.1f km/s, r_s = %.2f kpc"
      % (C.MW_DISCO["m"], C.MW_BULBO["m"], C.MW_NUCLEO["m"], C.MW_HALO["v_c"], C.MW_HALO["r_s"]))
print("LMC (tutorial): Hernquist %.2g Msun, a = %.0f kpc; hoy x = %s kpc, v = %s km/s" % (C.LMC_M, C.LMC_A, C.LMC_X0, C.LMC_V0))
print("Friccion dinamica: %s" % C.DF_LNL)
print("Sistema MW + LMC integrado %.0f Gyr hacia atras, dt = %.0f Myr; potenciales interpolados cada %.0f Myr"
      % (C.T_ATRAS / 1e3, C.DT_SISTEMA, C.DT_NUDOS))
print("Corrientes: FardalStreamDF, t_start = %.0f Gyr, dt = %.0f Myr, release_every = %d, semilla %d"
      % (C.T_START / 1e3, C.DT_CORRIENTE, C.RELEASE_EVERY, C.SEMILLA))
print("Sol (astropy v4.0): R = %.3f kpc, z = %.4f kpc, v = %s km/s" % (C.SOL_R, C.SOL_Z, C.SOL_V))
print("LSST: Dec < %.0f deg, r < %.1f (10 anios), PM r < %.1f, sigma_mu %s" % (C.LSST_DEC_MAX, C.LSST_R_LIM_10A, C.LSST_PM_R_MAX, C.LSST_PM_PUNTOS))
from p3streams import ejecutar
ejecutar.main(["planes"]);

MW (tutorial):  disco MN3 4.77e+10 Msun, bulbo 5e+09, nucleo 1.81e+09, halo NFW v_c = 171.6 km/s, r_s = 15.63 kpc
LMC (tutorial): Hernquist 1.8e+11 Msun, a = 20 kpc; hoy x = (-1.1, -41.1, -27.9) kpc, v = (-57.0, -226.0, 221.0) km/s
Friccion dinamica: {'L': 0.0, 'C': 1.22, 'alpha': 1.0, 'form': 'VdM'}
Sistema MW + LMC integrado 7 Gyr hacia atras, dt = 1 Myr; potenciales interpolados cada 5 Myr
Corrientes: FardalStreamDF, t_start = 5 Gyr, dt = 1 Myr, release_every = 2, semilla 2026
Sol (astropy v4.0): R = 8.122 kpc, z = 0.0208 kpc, v = (12.9, 245.6, 7.78) km/s
LSST: Dec < 5 deg, r < 27.0 (10 anios), PM r < 24.5, sigma_mu ((21.0, 0.2), (24.0, 1.0))
piloto       72 corrientes  12 polos x 2 sentidos x 3 familias (el 'empezar chico' del enunciado)
principal   288 corrientes  48 polos HEALPix x 3 familias x 2 fases
masa_1.0     36 corrientes  12 polos HEALPix x 3 familias, M_LMC = 1.0e+11 Msun
masa_1.5     36 corrientes  12 polos HEALPix x 3 familias, M_LMC = 1.5e+11 Msun
masa_1.8     36 corr